# Identify facility branches and locations

Read `records`, `raw_facilities` and `raw_pins` from notebook 01. Save the branch
entities, the link and review decisions, the postal directory by PIN and district,
and the raw FDR rows with an appended facility ID.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## Branch matching and geographic conflicts

A merge requires a unique same-name branch match with supporting address or
coordinate evidence. Review the candidate identifiers in ambiguous rows before
merging them. A postal-directory point describes an area. Facility coordinates
retain their own source and precision.

In [ ]:
from facility_tool.data import link_entities
from facility_tool.geo_audit import attach_routing_review, audit_locations
from facility_tool.geography import add_postal_context, build_postal_areas

records = STORE.read("records")
raw_fdr = STORE.read("raw_facilities")
entities, links = link_entities(records)
postal_areas = build_postal_areas(STORE.read("raw_pins"))
entities = add_postal_context(entities, postal_areas)
location_records, location_entity_review, location_quality_summary = audit_locations(
    records, entities, postal_areas, links=links, postal_offset_km=50,
)
entities = attach_routing_review(entities, location_entity_review)
snapshots = {"entities": entities, "links": links, "postal_areas": postal_areas,
             "location_records": location_records, "location_entity_review": location_entity_review,
             "location_quality_summary": location_quality_summary}
for name, frame in snapshots.items():
    STORE.write(name, frame)

decisions = records[["record_id", "source", "name"]].merge(links, on="record_id", validate="1:1")
show(decisions.groupby(["source", "link_status"]).size().rename("records").reset_index())
show(decisions.loc[decisions["link_status"].eq("ambiguous")])
show(entities.groupby(["location_routing_eligible", "location_hold_reason"]).size().rename("facilities").reset_index())

## Inspect locations that need a second source

Review the published address, PIN, original point and linked branch records.
The audit flags likely coordinate reversals, zero placeholders, shared points
and inconsistent or ambiguous PINs. Distance to a postal-office centroid
is a review signal. Shared points can describe a real campus. An explicit
conflict with all directory state labels blocks distance ranking for the branch.

In [ ]:
show(location_quality_summary.loc[location_quality_summary["count"].gt(0)])
show(location_records.loc[location_records["review_needed"].fillna(False).astype(bool)])
show(location_entity_review.loc[location_entity_review["review_needed"].fillna(False).astype(bool)])

## Map back to the original FDR rows

Source position ties each raw field to its transformed source row. An ambiguous
record stays in the raw audit with a blank facility ID. Branch-level statistics
and recommendations use links with an assigned ID.

In [ ]:
positions = records.loc[records["source"].eq("fdr"), ["record_id", "input_position"]]
mapping = positions.merge(links[["record_id", "facility_id"]], on="record_id", how="left", validate="1:1")
mapping["input_position"] = mapping["input_position"].astype(int)
if sorted(mapping["input_position"]) != list(range(len(raw_fdr))):
    raise ValueError("FDR records and raw rows differ. Rerun notebook 01 for this snapshot.")
linked = raw_fdr.copy()
linked["facility_id"] = mapping.sort_values("input_position")["facility_id"].fillna("").to_numpy()
STORE.write("linked_fdr", linked)
print(f"FDR rows: {len(linked):,} | Assigned branch ID: {linked['facility_id'].ne('').sum():,}")